# Rag from Scracth

In [38]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain python-dotenv

import os
from dotenv import load_dotenv

load_dotenv(override=True)

# LANGCHAIN_TRACING_V2: enables tracing of runs in LangSmith
# LANGSMITH_ENDPOINT: URL of the LangSmith service traces are sent to (depends on the ubication)
# LANGSMITH_API_KEY: credential used to authenticate traces in LangSmith
# OPENAI_API_KEY: credential used to authenticate requests to the OpenAI API
# (used for embeddings and/or chat completions in the RAG pipeline)

True

# Part 1: Overview

In [ ]:
import bs4
import requests
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

### Indexing

# Download a web article, for example The History of Sandwich
URL = "https://en.wikipedia.org/wiki/Sandwich"

# Fetch the raw HTML ourselves instead of via WebBaseLoader: that loader's
# bs_kwargs=dict(parse_only=bs4.SoupStrainer(...)) silently returns empty content
# on this bs4 version (SoupStrainer's class_ matching is broken at parse time),
# while the same class filter works fine with find_all() on an already-parsed tree.
response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
soup = bs4.BeautifulSoup(response.text, "html.parser")

# Keep only the article body and headings, dropping nav/infobox/categories/references
# noise that otherwise drowns out the real content during retrieval
content_tags = soup.find_all(class_=("mw-content-ltr", "mw-heading"))
page_text = "\n\n".join(tag.get_text(separator=" ", strip=True) for tag in content_tags)

docs = [Document(page_content=page_text, metadata={"source": URL})]


# Split the document
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=1000, # Each chunk 1000 characters
  chunk_overlap=200 # Each chunk repeat the 200 last characters of the previous one
)

splits = text_splitter.split_documents(docs) # List of chunks

# Embedding: Convert text into arrays
vectorstore = Chroma.from_documents(
  documents = splits,
  embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever() # Being able to ask questions


### Retrieval and generation

# Local copy of the "rlm/rag-prompt" template from the LangChain Hub.
# Defined here instead of pulled via Client().pull_prompt(...) because the Hub's public
# prompts live only in the US region and 404 from an EU-region LangSmith account.
prompt = ChatPromptTemplate.from_messages([
    ("human", """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:"""),
])

# Call the model (OpenAI one in this case)
llm = ChatOpenAI(
  model_name="gpt-5-nano", # NOV 2026 the more affordable OpenAI model is GPT 5 Nano
  temperature=0 # Want the model to be 0 creative
)


# Prepare the text
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
"""
question ──┬─► retriever ─► format_docs ──► context ─┐
           │                                         ├─► prompt ─► llm ─► text
           └─► (no changes) ────────────► question ──┘
"""

def ask(question):
    print(question)

    # LCEL pipeline ('|' chains Runnables, nothing runs until .invoke() below): the question
    # is fed in parallel into "context" (retriever fetches matching chunks, format_docs joins
    # them into one string) and "question" (passed through unchanged); that dict fills the
    # prompt template, the filled prompt goes to the llm, and StrOutputParser pulls out just
    # the plain text from the model's reply.
    rag_chain = (
      {"context": retriever | format_docs, "question": RunnablePassthrough()}
      | prompt
      | llm
      | StrOutputParser()
    )

    return rag_chain.invoke(question)

print(ask("What is a sandwitch?"))
print(ask("In which way is known sandwitch in spain?"))

What is a sandwitch
A sandwich is a dish typically made with bread and a filling such as meat, cheese, sauces, and vegetables, placed between slices of bread or on top of a slice (open sandwich). The bread serves as a container or wrapper for the filling. It originated as a portable Western-world food and has since become worldwide.
In which way is known sandwitch in spain?
In Spain, the word sandwich is borrowed from English and refers to a food item made with English sandwich bread; it is otherwise known as a bocadillo.


## Part 1.1: Indexing

In [40]:
# Example information
question = "What kinds of food do I like?"
document = "My favorite food is chocolate cake"

### Counting tokens

In [41]:
import tiktoken

def tokens_from_string(string: str, encoding: str) -> int:
  """Return the number of tokens `string` encodes into.

  Args:
    string: Text to tokenize.
    encoding: Name of the tiktoken encoding to use ("o200k_base" as is the one used by gpt5 nano).
  """
  encoding = tiktoken.get_encoding(encoding)
  num_tokens = len(encoding.encode(string))
  return num_tokens

num_tokens = tokens_from_string(document, "o200k_base")
print(f"Number of tokens in the document: {num_tokens}")

Number of tokens in the document: 6


### Embeddig generation using OpenAI API
The basis to make a RAG is transform information into arrays with the goal of compare their similarity.

In [42]:
from langchain_openai import OpenAIEmbeddings
embd = OpenAIEmbeddings()
query_result = embd.embed_query(question) # Vector representation of the question
document_result = embd.embed_query(document) # Vector representation of the document
len(query_result)

1536

### Cosine similarity between 2 arrays:
Measures how similar two vectors are by their **direction**, regardless of their length. It is the cosine of the angle $\theta$ between them.

 
$$
\cos(\theta) = \frac{A \cdot B}{\|A\|\,\|B\|}
$$
 

The closer the result is to 1, the more similar the vectors are.


What matters in an embedding is where the vector **points** (meaning), not how long it is. OpenAI embeddings are already normalized ($\|A\| = 1$), so the formula reduces to $A \cdot B$.

In [43]:
import numpy as np

def cosine_similarity(vec_a, vec_b):
    """Compute the cosine similarity between two vectors."""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot_product / (norm_a * norm_b)

similarity = cosine_similarity(query_result, document_result)
print(f"Cosine similarity between query and document: {similarity}")

Cosine similarity between query and document: 0.8320835197918032


### Load document

In [44]:
import bs4
import requests
from langchain_core.documents import Document

DOCUMENT_SOURCE = "https://en.wikipedia.org/wiki/Chocolate_cake"

# Same approach as Part 1: fetch the HTML ourselves and filter with find_all()
# instead of WebBaseLoader's bs_kwargs=dict(parse_only=bs4.SoupStrainer(...)),
# since SoupStrainer's class_ matching is broken at parse time on this bs4 version.
# Wikipedia pages use "mw-content-ltr"/"mw-heading" (the guide's "post-content" etc.
# only apply to that blog's HTML, not Wikipedia's).
cake_response = requests.get(DOCUMENT_SOURCE, headers={"User-Agent": "Mozilla/5.0"})
cake_soup = bs4.BeautifulSoup(cake_response.text, "html.parser")

cake_content_tags = cake_soup.find_all(class_=("mw-content-ltr", "mw-heading"))
cake_text = "\n\n".join(tag.get_text(separator=" ", strip=True) for tag in cake_content_tags)

blog_docs = [Document(page_content=cake_text, metadata={"source": DOCUMENT_SOURCE})]

### Splitter
Shorter pieces of information represent information in a better way. Also, LLM receives only useful chunks.

In [45]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

### Vectorstores
Save the chunks into a vector database (Chroma).
This is the indexation phase.

In [46]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

"""
The following function creates a vectorstore from the split documents
and calls an OpenAI embedding model in order to convert the text into arrays. The vectorstore is then used to retrieve relevant information based on a query.
"""
vectorstore = Chroma.from_documents(
  documents=splits,
  embedding=OpenAIEmbeddings())

# Object with friendly interface used to exeute queries
retriever = vectorstore.as_retriever()
docs = retriever.invoke("Tell me about chocolate cake")

len(docs) # Number of documents retrieved

4

## Part 1.2: Retrieval

In [47]:
# Index
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())


retriever = vectorstore.as_retriever(search_kwargs={"k": 1}) # Being k the k nearest neighbors to retrieve. In this case, we are retrieving the most relevant document based on the query.

docs = retriever.invoke("Tell me about chocolate cake first aparition")

len(docs) # Number of documents retrieved

1

## Part 1.4: Generation

### Defining a prompt using a template
Prompt is the piece that will join the LLM with the retriever

In [48]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

Get the OpenAI llm object

In [49]:
llm = ChatOpenAI(
  model_name="gpt-5-nano", # NOV 2026 the more affordable OpenAI model is GPT 5 Nano
  temperature=0 # Want the model to be 0 creative
)

### LangChain Expression Language
It functions like a terminal pipe: the output from the left side becomes the input for the right side.

In [50]:
# The "|" operator pipes the output of one component into the next:
#   input dict -> prompt (fills {context} and {question}) -> llm (generates the answer)
# "prompt" and "llm" are Runnables, so "|" returns a RunnableSequence,
# which exposes the standard interface: .invoke(), .stream(), .batch()...
chain = prompt | llm

# Run
chain.invoke({"context":docs,"question":"Where did the first appearance of chocolate cake occur?"})

AIMessage(content='The United States (America), in 1886.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 276, 'prompt_tokens': 370, 'total_tokens': 646, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Ej0ouJPQfeeeM9Qyt7Uew3uGyktxl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1ca5f-8788-76a4-a855-7518cae1b6a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 370, 'output_tokens': 276, 'total_tokens': 646, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 256}})

### Complete RAG Chain

In [51]:

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
      "context": retriever, # Question goes to retriever, who serch in Chroma in order to find relevant chunks
      "question": RunnablePassthrough() # Question goes without any changes to the prompt
    }
    | prompt # Prompt fills template with the context and the question using from_template
    | llm # LLM receives the filled prompt and generates an answer
    | StrOutputParser() # Answer is extracted from the LLM's output and returned as a string
)

rag_chain.invoke("Where did the first appearance of chocolate cake occur?")

'The United States.'